In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

In [3]:
HEADERS = {"User-Agent": "Mozilla/5.0 Chrome/124.0"}

def get_parcaqua_links():
    r = requests.get("https://www.parcs-aquatiques.com/parcs-aquatiques-france/", headers=HEADERS)
    soup = BeautifulSoup(r.text, "html.parser")
    parcaqua = {}
    for li in soup.select("ul li a[href*='parcs-aquatiques.com']"):
        name = li.get_text(strip=True)
        href = li["href"]
        if name and "/carte" not in href:
            parcaqua[name] = href
    return parcaqua

In [9]:
def extract_info(url):
    r = requests.get(url, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(r.text, "html.parser")

    adresse = None
    lat, lon = None, None

    # Chercher les balises <strong> qui contiennent "Adresse"
    for strong in soup.find_all("strong"):
        if "Adresse" in strong.get_text():
            # Le texte de l'adresse est dans le noeud suivant (NavigableString)
            next_node = strong.next_sibling
            if next_node:
                adresse = str(next_node).strip().lstrip(": ").strip()
            break

    # Coordonnées GPS
    gps_match = re.search(r"latitude\s*([\d.]+)\s*\|\s*longitude\s*([\d.]+)", soup.get_text())
    if gps_match:
        lat = float(gps_match.group(1))
        lon = float(gps_match.group(2))

    return {"adresse": adresse, "latitude": lat, "longitude": lon}

parc_aquatique = get_parcaqua_links()
records = []
for name, url in parc_aquatique.items():
    print(f"→ {name}")
    info = extract_info(url)
    records.append({"nom": name, "url": url, **info})
    time.sleep(0.8)

df = pd.DataFrame(records)
df.to_csv("parc_aquatique.csv", index=False, encoding="utf-8-sig")
print(df[["nom", "adresse", "latitude", "longitude"]])

→ Accueil
→ Promos
→ Carte France
→ Aqualand
→ Agen
→ Bassin d’Arcachon
→ Cap d’Agde
→ Fréjus
→ Port Leucate
→ Saint Cyprien
→ Saint Cyr sur Mer
→ Sainte Maxime
→ Aquasplash
→ Aquascope
→ Aquaboulevard
→ Aqua Béarnà Oloron Sainte Marie
→ Aquaboulevardà Paris
→ Aqua’Fun Park – Cobac Parcà Lanhélin
→ Aquajetà Narbonne Plage
→ Aqualand Agen– Walygator Sud Ouest
→ Aqualand Bassin d’Arcachon
→ Aqualand Fréjus
→ Aqualand Le Cap d’Agde
→ Aqualand Port Leucate
→ Aqualand Saint Cyprien
→ Aqualand Saint Cyr sur Mer
→ Aqualand Sainte Maxime
→ Aquaparc Isisà Dole
→ Aquascope Futuroscope
→ Aquasplashà Antibes
→ Aquatic Landesà Labenne-Océan
→ Atlantic Parkà Seignosse Océan
→ Atlantic Tobogganà Saint-Hilaire-de-Riez
→ Espace Grand Bleuà La Grande Motte
→ Iléosur l’île d’Oléron
→ Ludolacà Vesoul
→ Nyonsoleïadoà Nyons
→ O’Gliss Parken Vendée
→ Parc de la Bouscarasseà Serviers-et-Labaume
→ Pirates WorldCap d’Agde
→ Quercylandà Souillac
→ Vitam à Neydens
→ Wave Islandà Monteux
→ Western Parkà Biguglia
→

In [10]:
df

,nom,url,adresse,latitude,longitude
0,Accueil,https://www.parcs-aquatiques.com/,NaN,NaN,NaN
1,Promos,https://www.parcs-aquatiques.com/promo-billets...,NaN,NaN,NaN
2,Carte France,https://www.parcs-aquatiques.com/parcs-aquatiq...,NaN,NaN,NaN
3,Aqualand,https://www.parcs-aquatiques.com/promo-billet-...,NaN,NaN,NaN
4,Agen,https://www.parcs-aquatiques.com/aqualand-agen/,Château de Caudouin 47310 Roquefort,44.186173,0.579221
5,Bassin d’Arcachon,https://www.parcs-aquatiques.com/aqualand-bass...,route des lacs 33470 Gujan-Mestras,NaN,NaN
6,Cap d’Agde,https://www.parcs-aquatiques.com/aqualand-capd...,2 avenue des Iles d’Amérique 34300 Le Cap d’Agde,43.280843,3.494517
7,Fréjus,https://www.parcs-aquatiques.com/aqualand-frejus/,Quartier le Capou D559 83600 Fréjus,43.419434,6.728501
8,Port Leucate,https://www.parcs-aquatiques.com/aqualand-port...,avenue du Roussillon 11370 Port Leucate,42.842203,3.042208
9,Saint Cyprien,https://www.parcs-aquatiques.com/aqualand-sain...,avenue des Champs de Neptune 66750 Saint Cyprien,42.601210,3.032193


In [11]:
df = df.dropna(subset="adresse")

In [12]:
df

,nom,url,adresse,latitude,longitude
4,Agen,https://www.parcs-aquatiques.com/aqualand-agen/,Château de Caudouin 47310 Roquefort,44.186173,0.579221
5,Bassin d’Arcachon,https://www.parcs-aquatiques.com/aqualand-bass...,route des lacs 33470 Gujan-Mestras,NaN,NaN
6,Cap d’Agde,https://www.parcs-aquatiques.com/aqualand-capd...,2 avenue des Iles d’Amérique 34300 Le Cap d’Agde,43.280843,3.494517
7,Fréjus,https://www.parcs-aquatiques.com/aqualand-frejus/,Quartier le Capou D559 83600 Fréjus,43.419434,6.728501
8,Port Leucate,https://www.parcs-aquatiques.com/aqualand-port...,avenue du Roussillon 11370 Port Leucate,42.842203,3.042208
9,Saint Cyprien,https://www.parcs-aquatiques.com/aqualand-sain...,avenue des Champs de Neptune 66750 Saint Cyprien,42.601210,3.032193
10,Saint Cyr sur Mer,https://www.parcs-aquatiques.com/aqualand-sain...,ZAC des Pradeaux 83270 Saint-Cyr-sur-Mer,43.184257,5.692833
11,Sainte Maxime,https://www.parcs-aquatiques.com/aqualand-sain...,Avenue Gaston Rebuffat 83120 Sainte Maxime,43.327553,6.619147
12,Aquasplash,https://www.parcs-aquatiques.com/aquasplash-an...,306 avenue Mozart 06600 Antibes,43.614914,7.120413
13,Aquascope,https://www.parcs-aquatiques.com/aquascope-fut...,Avenue René Monory 86360 Chasseneuil-du-Poitou,46.663320,0.361748


In [13]:
df.to_csv("parc_aquatique.csv")